# CVE Feature Engineering & Weak Label Construction

**Purpose**: Transform enriched CVE data into training-ready features and weak supervision labels

**What this notebook does**:
1. **Feature Verification** - Validate complete 53-feature set loaded from database
2. **Weak Label Construction** - Build graded labels (0-3) with confidence scores
3. **Label Diagnostics** - Analyze label quality and distribution
4. **Feature Analysis** - Completeness, correlations, and quality checks
5. **Data Preparation** - Save processed features for STEP_5 model training

**Pipeline Alignment**:
- Upstream: STEP_1 and STEP_3 populate/enrich feature columns in database
- Current: STEP_4 validates and labels the full feature matrix
- Downstream: STEP_5 trains models and runs `scripts/run_scientific_protocol.py`

**Key Innovation**: Confidence-weighted weak supervision
- Each label has a confidence score (0-1) based on signal reliability
- KEV flags have highest confidence (1.0)
- Heuristic signals have lower confidence (0.3-0.7)
- Model training uses confidence-weighted ranking loss

---

## 1. Setup & Imports

In [ ]:
import sys
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))
os.chdir(project_root)

# Import project modules
from src.core.cve_database import CVEDatabase
from src.features.engineering import create_all_features, get_default_feature_cols
from src.features.labeling import build_weak_labels, print_label_diagnostics
from src.utils.notebook_helpers import save_plot, save_dataframe, display_sample, setup_notebook_output
from config.settings import settings

# Configure notebook display
setup_notebook_output()

print(f"[OK] Project root: {project_root}")
print(f"[OK] Imports successful")

## 2. Load Raw CVE Data

In [ ]:
# Load CVEs with ALL enrichments from database (basic + enhanced features)
db = CVEDatabase()

query = """
SELECT 
    c.cve_id,
    c.published,
    c.modified,
    c.cvss,
    c.cvss_vector,
    c.cwe,
    -- Basic enrichments (16 features)
    e.kev_flag,
    e.epss_score,
    e.epss_percentile,
    e.is_healthcare,
    e.healthcare_score,
    e.attack_flag,
    e.attack_technique_count,
    e.chpl_flag,
    e.is_curated,
    e.curated_severity,
    -- CVSS Decomposition (10 features)
    e.cvss_av,
    e.cvss_ac,
    e.cvss_pr,
    e.cvss_ui,
    e.cvss_s,
    e.cvss_c,
    e.cvss_i,
    e.cvss_a,
    e.cvss_score_derived,
    e.cvss_severity_category,
    -- CWE Intelligence (8 features)
    e.cwe_is_top25,
    e.cwe_is_injection,
    e.cwe_is_crypto,
    e.cwe_is_access_control,
    e.cwe_is_input_validation,
    e.cwe_is_memory_corruption,
    e.cwe_category,
    e.cwe_severity_score,
    -- Description NLP (10 features)
    e.desc_has_rce,
    e.desc_has_auth_bypass,
    e.desc_has_priv_esc,
    e.desc_has_sqli,
    e.desc_has_xss,
    e.desc_has_dos,
    e.desc_has_buffer_overflow,
    e.desc_has_path_traversal,
    e.desc_has_csrf,
    e.desc_has_xxe,
    -- Vendor Features (3 features)
    e.vendor_is_high_risk,
    e.vendor_is_healthcare,
    e.vendor_risk_score,
    -- Interaction Features (6 features)
    e.ultimate_risk,
    e.critical_exploitable,
    e.network_accessible,
    e.auth_not_required,
    e.high_impact_network,
    e.healthcare_critical
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.cvss IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])
df['modified'] = pd.to_datetime(df['modified'])

# Define all feature columns (53 total: 16 basic + 37 enhanced)
basic_features = [
    'kev_flag', 'epss_score', 'epss_percentile', 
    'is_healthcare', 'healthcare_score',
    'attack_flag', 'attack_technique_count',
    'chpl_flag', 'is_curated', 'curated_severity'
]

cvss_features = [
    'cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s',
    'cvss_c', 'cvss_i', 'cvss_a', 'cvss_score_derived', 'cvss_severity_category'
]

cwe_features = [
    'cwe_is_top25', 'cwe_is_injection', 'cwe_is_crypto',
    'cwe_is_access_control', 'cwe_is_input_validation',
    'cwe_is_memory_corruption', 'cwe_category', 'cwe_severity_score'
]

nlp_features = [
    'desc_has_rce', 'desc_has_auth_bypass', 'desc_has_priv_esc',
    'desc_has_sqli', 'desc_has_xss', 'desc_has_dos',
    'desc_has_buffer_overflow', 'desc_has_path_traversal',
    'desc_has_csrf', 'desc_has_xxe'
]

vendor_features = [
    'vendor_is_high_risk', 'vendor_is_healthcare', 'vendor_risk_score'
]

interaction_features = [
    'ultimate_risk', 'critical_exploitable', 'network_accessible',
    'auth_not_required', 'high_impact_network', 'healthcare_critical'
]

all_features = basic_features + cvss_features + cwe_features + nlp_features + vendor_features + interaction_features

print(f"\n{'='*70}")
print("DATA LOADED FROM DATABASE")
print(f"{'='*70}")
print(f"Total CVEs: {len(df):,}")
print(f"Date range: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"\n All Features (53 total):")
print(f"  Basic enrichments: {len(basic_features)}")
print(f"  CVSS decomposition: {len(cvss_features)}")
print(f"  CWE intelligence: {len(cwe_features)}")
print(f"  Description NLP: {len(nlp_features)}")
print(f"  Vendor features: {len(vendor_features)}")
print(f"  Interaction features: {len(interaction_features)}")
print(f"{'='*70}\n")

display_sample(df[['cve_id', 'published', 'cvss'] + basic_features[:5]], n=10, title="Sample Raw Data")

## 3. Feature Verification

Verify all 53 features are loaded from database (16 basic + 37 enhanced) 

In [ ]:
# Features are already loaded from database - just verify completeness
df_features = df.copy()

print(f"\n{'='*70}")
print("FEATURE VERIFICATION")
print(f"{'='*70}")
print(f"Total CVEs: {len(df_features):,}")
print(f"Total features: {len(all_features)}")
print(f"{'='*70}\n")

# Show feature breakdown by category
print(f"[STATS] Feature Categories:")
print(f"\n1. Basic Enrichments ({len(basic_features)} features):")
for i, col in enumerate(basic_features, 1):
    print(f"  {i:2d}. {col}")

print(f"\n2. CVSS Decomposition ({len(cvss_features)} features):")
for i, col in enumerate(cvss_features, 1):
    print(f"  {i:2d}. {col}")

print(f"\n3. CWE Intelligence ({len(cwe_features)} features):")
for i, col in enumerate(cwe_features, 1):
    print(f"  {i:2d}. {col}")

print(f"\n4. Description NLP ({len(nlp_features)} features):")
for i, col in enumerate(nlp_features, 1):
    print(f"  {i:2d}. {col}")

print(f"\n5. Vendor Features ({len(vendor_features)} features):")
for i, col in enumerate(vendor_features, 1):
    print(f"  {i:2d}. {col}")

print(f"\n6. Interaction Features ({len(interaction_features)} features):")
for i, col in enumerate(interaction_features, 1):
    print(f"  {i:2d}. {col}")

display_sample(df_features[['cve_id', 'cvss'] + all_features[:10]], n=10, title="Sample Features")

## 4. Feature Completeness Analysis

In [ ]:
# Analyze completeness (NULL values) in features
print(f"\n{'='*70}")
print("FEATURE COMPLETENESS REPORT")
print(f"{'='*70}")

# Calculate completeness for each feature
missing_report = {}
for col in all_features:
    if col in df_features.columns:
        total = len(df_features)
        missing = df_features[col].isna().sum()
        pct_complete = ((total - missing) / total) * 100
        missing_report[col] = {
            'missing': missing,
            'complete': total - missing,
            'pct_complete': pct_complete
        }

# Sort by completeness
sorted_features = sorted(missing_report.items(), key=lambda x: x[1]['pct_complete'])

print(f"\n[WARN] Features with missing values:")
has_missing = False
for feat, stats in sorted_features:
    if stats['pct_complete'] < 100:
        status = "[WARN]" if stats['pct_complete'] >= 70 else "[FAIL]"
        print(f"  {status} {feat:35s}: {stats['pct_complete']:5.1f}% complete ({stats['missing']:,} missing)")
        has_missing = True

if not has_missing:
    print("  [OK] All features are 100% complete!")

print(f"\n[OK] Complete features (100%):")
complete_count = sum(1 for f, s in sorted_features if s['pct_complete'] == 100)
print(f"  {complete_count} / {len(all_features)} features")

In [ ]:
# Visualize feature completeness
completeness = {}
for col in all_features:
    if col in df_features.columns:
        completeness[col] = (df_features[col].notna().sum() / len(df_features)) * 100

completeness_df = pd.DataFrame({
    'Feature': list(completeness.keys()),
    'Completeness': list(completeness.values())
}).sort_values('Completeness')

fig = px.bar(
    completeness_df,
    x='Completeness',
    y='Feature',
    orientation='h',
    title=f'Feature Completeness (%) - {len(all_features)} Total Features',
    labels={'Completeness': 'Completeness (%)', 'Feature': ''},
    color='Completeness',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(height=max(600, len(all_features) * 15))

save_plot(fig, 'feature_completeness')

print(f"\n[OK] Feature completeness plot saved")
print(f"  Average completeness: {completeness_df['Completeness'].mean():.1f}%")

## 5. Weak Label Construction

Build graded labels (0-5) with confidence scores based on multiple risk signals

In [ ]:
# Construct weak labels using modular function
print("Constructing weak labels...")
df_labeled = build_weak_labels(df_features)

print(f"\n{'='*70}")
print("WEAK LABEL CONSTRUCTION COMPLETE")
print(f"{'='*70}")
print(f"Total CVEs: {len(df_labeled):,}")
print(f"Labeled CVEs: {df_labeled['soft_label'].notna().sum():,}")
print(f"{'='*70}\n")

# Label distribution
label_dist = df_labeled['soft_label'].value_counts().sort_index()
print(f"[STATS] Label Distribution:")
for label, count in label_dist.items():
    pct = (count / len(df_labeled)) * 100
    print(f"  Label {label}: {count:7,} ({pct:5.2f}%)")

# Confidence stats
print(f"\n Confidence Statistics:")
print(f"  Mean: {df_labeled['label_confidence'].mean():.3f}")
print(f"  Median: {df_labeled['label_confidence'].median():.3f}")
print(f"  High confidence (≥0.7): {(df_labeled['label_confidence'] >= 0.7).sum():,} ({(df_labeled['label_confidence'] >= 0.7).mean()*100:.1f}%)")

## 6. Label Diagnostics

In [ ]:
# Print comprehensive label diagnostics
print_label_diagnostics(df_labeled)

In [ ]:
# Visualize label distribution
label_counts = df_labeled['soft_label'].value_counts().sort_index()

fig = px.bar(
    x=label_counts.index.astype(str),
    y=label_counts.values,
    title='Weak Label Distribution',
    labels={'x': 'Label (0=Low Risk, 3=Critical)', 'y': 'Number of CVEs'},
    text=label_counts.values,
    color=label_counts.values,
    color_continuous_scale='YlOrRd'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=450, showlegend=False)

save_plot(fig, 'weak_label_distribution')

print("\n[OK] Label distribution plot saved")

In [ ]:
# Analyze confidence distribution
fig = px.histogram(
    df_labeled,
    x='label_confidence',
    nbins=50,
    title='Label Confidence Distribution',
    labels={'label_confidence': 'Confidence Score', 'count': 'Number of CVEs'},
    color_discrete_sequence=['#3498DB']
)
fig.update_layout(height=400)

save_plot(fig, 'label_confidence_distribution')

print(f"\n[STATS] Confidence Statistics:")
print(f"  Mean: {df_labeled['label_confidence'].mean():.3f}")
print(f"  Median: {df_labeled['label_confidence'].median():.3f}")
print(f"  High confidence (>0.8): {(df_labeled['label_confidence'] > 0.8).sum():,} ({(df_labeled['label_confidence'] > 0.8).mean()*100:.1f}%)")
print(f"  Low confidence (<0.5): {(df_labeled['label_confidence'] < 0.5).sum():,} ({(df_labeled['label_confidence'] < 0.5).mean()*100:.1f}%)")

## 7. Feature Correlations with Labels

In [ ]:
# Calculate correlations between features and labels
# Use all numeric features from all_features list
numeric_feature_cols = []
for col in all_features:
    if col in df_labeled.columns and df_labeled[col].dtype in ['int64', 'float64', 'int32', 'float32']:
        numeric_feature_cols.append(col)

print(f"\n Analyzing {len(numeric_feature_cols)} numeric features...")

if 'soft_label' in df_labeled.columns:
    # Calculate correlations
    correlations = df_labeled[numeric_feature_cols + ['soft_label']].corr()['soft_label'].drop('soft_label')
    
    # Remove NaN correlations (features with no variance)
    valid_correlations = correlations.dropna().sort_values(ascending=False)
    
    # Identify problematic features
    nan_features = correlations[correlations.isna()].index.tolist()
    
    print(f"\n{'='*70}")
    print("FEATURE-LABEL CORRELATIONS")
    print(f"{'='*70}")
    
    if nan_features:
        print(f"\n[WARN] Features with no variance (excluded from correlation):")
        for feat in nan_features:
            unique_vals = df_labeled[feat].nunique()
            print(f"  • {feat}: {unique_vals} unique value(s)")
        print()
    
    print(f"Top 15 positively correlated features:")
    for feat, corr in valid_correlations.head(15).items():
        print(f"  {feat:35s}: {corr:+.3f}")
    
    if len(valid_correlations) > 15:
        print(f"\nTop 10 negatively correlated features:")
        for feat, corr in valid_correlations.tail(10).items():
            print(f"  {feat:35s}: {corr:+.3f}")
    print(f"{'='*70}\n")
    
    # Visualize top correlations (only valid ones)
    top_corrs = pd.concat([valid_correlations.head(15), valid_correlations.tail(10)]).sort_values()
    
    fig = px.bar(
        x=top_corrs.values,
        y=top_corrs.index,
        orientation='h',
        title='Feature Correlations with Label (Top 25)',
        labels={'x': 'Correlation', 'y': 'Feature'},
        color=top_corrs.values,
        color_continuous_scale='RdBu_r'
    )
    fig.update_layout(height=700, showlegend=False)
    
    save_plot(fig, 'feature_label_correlations')
    print("[OK] Feature-label correlation plot saved")

## 8. Label Quality Analysis

In [ ]:
# Analyze label quality by checking signal combinations
print(f"\n{'='*70}")
print("LABEL QUALITY ANALYSIS")
print(f"{'='*70}")

# High confidence, high priority labels
high_priority = df_labeled[(df_labeled['soft_label'] >= 2) & (df_labeled['label_confidence'] > 0.8)]
print(f"\n High Priority CVEs (label≥2, confidence>0.8):")
print(f"   Count: {len(high_priority):,}")
print(f"   KEV: {high_priority['kev_flag'].sum():,} ({high_priority['kev_flag'].mean()*100:.1f}%)")
print(f"   Healthcare: {high_priority['is_healthcare'].sum():,} ({high_priority['is_healthcare'].mean()*100:.1f}%)")
print(f"   Mean CVSS: {high_priority['cvss'].mean():.2f}")

# Low confidence labels (need manual review)
low_confidence = df_labeled[df_labeled['label_confidence'] < 0.4]
print(f"\n[WARN]  Low Confidence CVEs (confidence<0.4):")
print(f"   Count: {len(low_confidence):,}")
print(f"   Label distribution: {dict(low_confidence['soft_label'].value_counts().sort_index())}")

# Multi-signal CVEs (most reliable)
multi_signal = df_labeled[
    (df_labeled['kev_flag'] == 1) | 
    (df_labeled['is_healthcare'] == 1) | 
    (df_labeled['attack_flag'] == 1)
]
print(f"\n[OK] Multi-Signal CVEs (KEV or Healthcare or ATT&CK):")
print(f"   Count: {len(multi_signal):,}")
print(f"   Mean label: {multi_signal['soft_label'].mean():.2f}")
print(f"   Mean confidence: {multi_signal['label_confidence'].mean():.3f}")

print(f"{'='*70}\n")

## 9. Save Processed Features

In [ ]:
# Save processed features with labels for model training
output_path = save_dataframe(
    df_labeled,
    name=f'features_with_labels_{pd.Timestamp.now().strftime("%Y%m%d")}',
    subdir='features',
    format='csv'  # CSV format for compatibility
)

print(f"\n[OK] Processed features saved: {output_path}")
print(f"  Rows: {len(df_labeled):,}")
print(f"  Columns: {len(df_labeled.columns)}")
print(f"  File size: {output_path.stat().st_size / (1024**2):.2f} MB")

## 10. Feature Summary Statistics

In [ ]:
# Generate summary statistics for all features
print(f"\n{'='*70}")
print("FEATURE ENGINEERING SUMMARY")
print(f"{'='*70}")

print(f"\n[STATS] Dataset:")
print(f"  Total CVEs: {len(df_labeled):,}")
print(f"  Total features: {len(all_features)} (16 basic + 37 enhanced)")
print(f"    - Basic enrichments: {len(basic_features)}")
print(f"    - CVSS decomposition: {len(cvss_features)}")
print(f"    - CWE intelligence: {len(cwe_features)}")
print(f"    - Description NLP: {len(nlp_features)}")
print(f"    - Vendor features: {len(vendor_features)}")
print(f"    - Interaction features: {len(interaction_features)}")
print(f"  Date range: {df_labeled['published'].min().date()} to {df_labeled['published'].max().date()}")

print(f"\n  Weak Labels:")
print(f"  Labels: 0 (Low) to 3 (Critical)")
print(f"  Mean label: {df_labeled['soft_label'].mean():.2f}")
print(f"  Mean confidence: {df_labeled['label_confidence'].mean():.3f}")
print(f"  High priority (label≥2): {(df_labeled['soft_label'] >= 2).sum():,} ({(df_labeled['soft_label'] >= 2).mean()*100:.1f}%)")

print(f"\n Feature Completeness:")
avg_completeness = completeness_df['Completeness'].mean()
print(f"  Average: {avg_completeness:.1f}%")
print(f"  Complete features (100%): {(completeness_df['Completeness'] == 100).sum()}")
print(f"  Partial features (90-99%): {((completeness_df['Completeness'] >= 90) & (completeness_df['Completeness'] < 100)).sum()}")
print(f"  Incomplete features (<90%): {(completeness_df['Completeness'] < 90).sum()}")

print(f"\n Outputs:")
print(f"  Features: outputs/features/features_with_labels_*.csv")
print(f"  Plots: outputs/plots/")

print(f"\n{'='*70}")

# Close database
db.conn.close()
print("\n[OK] Feature Engineering Complete")

## Next Steps

1. **Model Training + Protocol** -> Run `STEP_4_All_Models_Training.ipynb` to:
   - Train confidence-weighted LambdaMART and baseline models
   - Execute unified protocol: `python scripts/run_scientific_protocol.py`
   - Load protocol artifacts from `outputs/scientific_protocol/`
   - Compare complete split and year-based split results reproducibly

2. **Model Comparison + Advanced Analysis** -> Run `STEP_5_Model_Comparison_And_Evaluation.ipynb` using STEP_4 artifacts

---